# ramanujan-qft research-alpha demonstration

Candidate **0.3.0a1**. Run every cell in order after installing the package.
This notebook uses base dependencies only; no SciPy or repository-only imports.

The examples use positive Euclidean propagators and the loop measure
$d^2k\,d^2\ell/\pi^2$. Input masses are **squared** masses.
The generated RMT series and equal-mass Taylor continuation are distinct APIs.
See `docs/CAPABILITIES.md` for scope and `docs/SCIENTIFIC_REVIEW.md` for review status.


In [1]:
import mpmath as mp
from ramanujan_qft import (
    __version__, ScalarEdge, ScalarGraph, compile_sunset_rmt,
    integrate_graph_rmt, integrate_sunset_taylor, RMTDomainError,
)
print("Package version:", __version__)
def sunset(momentum):
    return ScalarGraph([ScalarEdge(0, 1, 1)]*3,
                       {0: [momentum], 1: [-momentum]})


Package version: 0.3.0a1


## Graph to exact bracket families to a convergent RMT value

At $p_E=10$, unit masses satisfy $p_E>m_1+m_2+m_3$.
The returned tail bound concerns the mathematical series. Arithmetic error
estimates are numerical diagnostics, not rigorous interval enclosures.


In [2]:
series = compile_sunset_rmt(sunset(10))
print("Exponent matrix:", series.bracket_system.matrix)
print("Regulated Gamma families:", len(series.gamma_terms))
rmt = series.evaluate(dps=30)
print("Value:", mp.nstr(rmt.value, 30))
print("Method:", rmt.method)
assert mp.nstr(rmt.value, 30) == "0.629016707590614149590827253884"


Exponent matrix: Matrix([[1, 1, 0, 0, 1, 1, 0], [1, 0, 1, 0, 1, 0, 1], [1, 0, 0, 1, 0, 1, 1], [1, 0, 0, 0, 1, 1, 1]])
Regulated Gamma families: 8
Value: 0.629016707590614149590827253884
Method: multivariate RMT / coalesced Gamma series


## Equal-mass vacuum from the separate Taylor method

The boundary value is $4\operatorname{Cl}_2(\pi/3)/\sqrt3$.
The transformed Taylor series extends it to finite Euclidean momenta.


In [3]:
vacuum = integrate_sunset_taylor(sunset(0), dps=30)
print("Vacuum:", mp.nstr(vacuum.value, 30))
print("Method:", vacuum.method)
with mp.workdps(100):
    clausen = 4*mp.clsin(2, mp.pi/3)/mp.sqrt(3)
    assert abs(vacuum.value-clausen) <= vacuum.error_estimate


Vacuum: 2.34390723868945889060156228887
Method: equal-mass sunset / transformed Taylor series


## Independent parameter-integral check at low momentum

Set $x+y=1$ and analytically integrate the remaining positive quadratic in $z$.
This reference uses neither the Taylor recurrence nor its Clausen seed.


In [4]:
low = integrate_sunset_taylor(sunset(1), dps=30)
with mp.workdps(45):
    def parameter_integrand(x):
        a = x*(1-x)
        if not a:
            return mp.mpf(0)
        delta = (1-a)**2 + a*(2*(1+a)+a)
        root = mp.sqrt(delta)
        return 2*mp.asinh(root/(2*mp.sqrt(a)))/root
    reference = mp.quad(parameter_integrand, [0, mp.mpf('.5'), 1])
    assert abs(low.value-reference) <= low.error_estimate + mp.mpf('1e-40')
print("Low momentum:", mp.nstr(low.value, 30))
print("Independent reference:", mp.nstr(reference, 30))


Low momentum: 2.23679270021264651052926362688
Independent reference: 2.23679270021264651052926362688


## Overlap between the two methods

At $p_E=6$, both the equal-mass Taylor series and the generated RMT series apply.


In [5]:
a = integrate_graph_rmt(sunset(6), dps=23)
b = integrate_sunset_taylor(sunset(6), dps=23)
with mp.workdps(50):
    assert abs(a.value-b.value) <= a.error_estimate+b.error_estimate
print("RMT:", mp.nstr(a.value, 23))
print("Taylor:", mp.nstr(b.value, 23))


RMT: 1.0523540707776368857028
Taylor: 1.0523540707776368857028


## Explicit method boundaries

An RMT request outside its charts stays an RMT-domain refusal. The package
does not silently replace it with a Taylor or quadrature calculation.


In [6]:
try:
    compile_sunset_rmt(sunset(0))
except RMTDomainError:
    print("Equal-mass vacuum is outside the RMT charts; choose the Taylor API explicitly.")
else:
    raise AssertionError("Expected an RMT-domain refusal")


Equal-mass vacuum is outside the RMT charts; choose the Taylor API explicitly.


## Reproduce and cite

From a source checkout, run `python tools/audit_sunset.py` for the fixed-seed
benchmark grid and `python -m pytest -q` for the full tests. Source hashes and
reference precision checks are included in the audit report.

Cite the software version using `CITATION.cff`, plus the original methods:
[González–Moll–Schmidt](https://arxiv.org/abs/1103.0588),
[Ananthanarayan–Friot–Ghosh](https://arxiv.org/abs/1911.10096),
[Laporta–Remiddi](https://arxiv.org/abs/hep-ph/0406160), and
[Müller-Stach–Weinzierl–Zayadeh](https://arxiv.org/abs/1112.4360).
External scientific review and public release are separate outstanding steps.
